# Satellite Imagery Pipeline

## Trabalho prático para disciplina de Processamento de Imagens

> Desenvolvido por Murilo M. Grosso e Octávio X. Fúrio

### Pipeline setup

In [42]:
ALIST_PSIZE = 1728
CHANNEL_SNR = 1     # dB: Realistically, 1dB
MAX_ITERS   = 50    # More are better & slower

In [51]:
from pathlib import Path
import subprocess

# g++ src/*.cpp -o bin/main -I includes/
subprocess.run("g++ src/*.cpp -o ./jpeg -I includes/", shell=True)

Path("inputs").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

# Extras
Path("pipeline/res_imgs").mkdir(parents=True, exist_ok=True)

# Path("metadata").mkdir(parents=True, exist_ok=True)

### Compress the input images

In [ ]:
# Resize images to nearest multiple of alist size
# Done to facilitate block division later on
import os
import numpy as np
from PIL import Image

def get_new_size(w, h, alist_s):
    ar_orig = w / h
    best, best_error = None, float('inf')
    for new_h in range(max(1, h - 50), h + 50):
        for new_w in range(max(1, w - 50), w + 50):
            # res * channels * bits/byte
            if (new_w * new_h * 3 * 8) % alist_s == 0:
                error = abs((new_w / new_h) - ar_orig)
                if error < best_error:
                    best_error = error
                    best = (new_w, new_h)
    return best

for fname in os.listdir("inputs"):
    img = Image.open(os.path.join("inputs", fname))
    nw, nh = get_new_size(*img.size, ALIST_PSIZE)     # type: ignore
    img_resized = img.resize((nw, nh), Image.LANCZOS)   # type: ignore
    img_resized.save(os.path.join("pipeline/res_imgs", fname))
    if      nw * nh * 3 * 8 / ALIST_PSIZE != \
        int(nw * nh * 3 * 8 / ALIST_PSIZE):
        print(f"{fname} could not be properly resized...")
        continue    
    print(f"{fname} resized from {img.size} to {(nw, nh)}. To be transmited over {nw * nh * 3 * 8 / ALIST_PSIZE:.3f} packets")

1.png resized from (482, 498) to (456, 471). Will be transmited over 2983.000 packets


In [ ]:
for im in os.listdir("inputs"):
    # bin/main 0 image_name.png bitstream.data
    subprocess.run(
        f"jpeg.exe 0 inputs/{im} outputs/_{im.split('.')[0]}.data",
        shell=True
    )